# Model 1 - Perseptron

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import keras
from keras.models import Sequential
from keras.layers import Dense
from sklearn.model_selection import train_test_split
import random

tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

In [2]:
customers_data = pd.read_excel("../Datasets/cleaned_customers_churn_data.xlsx")
customers_data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Germany,Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,0,0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0,1
2,502,0,42,8,159660.80,3,1,0,113931.57,1,0,0
3,699,0,39,1,0.00,2,0,0,93826.63,0,0,0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0,1


In [3]:
x = customers_data.drop(columns=['Exited'])
y = customers_data['Exited']

In [4]:
x_train,x_test, y_train, y_test = train_test_split(x,y,test_size=0.2, random_state=42)


In [5]:
x_train.shape, y_train.shape, x_test.shape, y_test.shape

((8000, 11), (8000,), (2000, 11), (2000,))

In [6]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [7]:
# Design
p_model = Sequential()
p_model.add(Dense(input_dim=11, units=1,activation="sigmoid", kernel_initializer="uniform"))

# Compile
p_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# fit
p_model.fit(x_train, y_train, batch_size=32, epochs=100)

# predict
y_pred_train = p_model.predict(x_train)
y_pred_train = (y_pred_train > 0.5)

# evaluate
from sklearn.metrics import accuracy_score
print("Training Accuracy: ", accuracy_score(y_train, y_pred_train))

# predict on test data
y_pred_test = p_model.predict(x_test)
y_pred_test = (y_pred_test > 0.5)

# evaluate on test data
print("Test Accuracy: ", accuracy_score(y_test, y_pred_test))


Epoch 1/100


c:\softwers\anaconda\installed_here\envs\churn_env\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 695us/step - accuracy: 0.7151 - loss: 0.6430
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 594us/step - accuracy: 0.7857 - loss: 0.5691
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 579us/step - accuracy: 0.7979 - loss: 0.5238
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 586us/step - accuracy: 0.8009 - loss: 0.4937
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 594us/step - accuracy: 0.8018 - loss: 0.4734
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 604us/step - accuracy: 0.8060 - loss: 0.4595
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 594us/step - accuracy: 0.8081 - loss: 0.4500
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 596us/step - accuracy: 0.8086 - loss: 0.4437
Epoch 9/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 580us/step - accuracy: 0.8098 - loss: 0.4394
Epoch 10/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 623us/step - accuracy: 0.8090 - loss: 0.4366
Epoch 11/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 627us/step - accuracy: 0.8098 - loss: 0.4348
Epoch 12/100
250/250 ━━━━━━━━━━

In [8]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score    
)

print(confusion_matrix(y_test, y_pred_test))
print(classification_report(y_test, y_pred_test))
print("ROC AUC Score:", roc_auc_score(y_test, y_pred_test))


[[1543   64]
 [ 312   81]]
              precision    recall  f1-score   support

           0       0.83      0.96      0.89      1607
           1       0.56      0.21      0.30       393

    accuracy                           0.81      2000
   macro avg       0.70      0.58      0.60      2000
weighted avg       0.78      0.81      0.78      2000

ROC AUC Score: 0.5831405539695131


# Perceptron Model (Single Layer Neural Network) - Observation
- Built a baseline binary classification model using a single neuron with sigmoid activation.
- Achieved 81.2% training accuracy and 81.2% testing accuracy, indicating no major overfitting.
- The model performed well in identifying customers who stay with the bank (Recall for Class 0 = 96%).
- However, it struggled to identify customers who churn (Recall for Class 1 = 21%), missing a large number of actual churn customers (FN = 312).
- Precision for churn class = 56%, meaning when the model predicted churn, it was correct 56% of the time.
- F1-Score for churn class = 0.30, indicating a poor balance between Precision and Recall.
- ROC-AUC Score = 0.58, which is only slightly better than random guessing (0.50).
- The model correctly predicts most customers who stay with the bank but fails to identify many customers who are likely to churn. Since the primary business objective is to detect potential churners and take retention actions, improving Recall and F1-Score for the churn class should be the focus of the next model.
- Overall, the perceptron serves as a good baseline model, but its ability to detect churn customers is weak. More complex models such as Multi-Layer Perceptron (MLP), Random Forest, or XGBoost may improve Recall, F1-Score, and ROC-AUC.